## TEST STABLE STATES

In [ ]:
import time
import matplotlib.pyplot as plt
from sympy import SOPform, simplify_logic
from tabulate import tabulate
from boon import *
import pprint
import z3

## Load File

In [ ]:
boon = BooN.from_textfile("Example/taboonbest.bnet")
pprint.pprint(boon, width=1) 

In [ ]:
print("Variables", boon.variables)
print("Size",len(boon.variables))

## Stable state generation

In [ ]:
print("Stable states:")
stable  = boon.stable_states

In [ ]:
print(tabulate(stable, headers='keys', tablefmt='dpsl'))

In [ ]:
print("# Stable states: %d"% len(stable))

### Hand made computation

In [ ]:
constraint = boon.stability_constraints()
print(constraint)

In [ ]:
solver = z3.Solver()
solver.add(logic.sympy2z3(boon.stability_constraints()))

all_vars = [z3.Bool(str(var)) for var in boon.variables]
models = []
while solver.check() == z3.sat:
    model = solver.model()
    models.append(model)
    block = [v != model.eval(v, model_completion=True) for v in all_vars]
    solver.add(z3.Or(block))

stable2 = list(map(lambda model: {symbols(str(v)): bool(model.eval(v, model_completion=True)) for v in all_vars}, models))
print(f"Stable states: {len(stable2)}")

In [ ]:
print(tabulate(stable2, headers='keys', tablefmt='dpsl'))

In [ ]:
print("# Stable states: %d"% len(stable2))

### Autre version sans assignation de toutes les variables mais force l'assignation

In [ ]:
solver = z3.Solver()  # initialize z3 solver
solver.add(logic.sympy2z3(boon.stability_constraints()))  # add stability constraints translated in z3.

# Enumerate all models
models = []
while solver.check() == z3.sat:
    model = solver.model()
    models.append(model)
    # Block the current model to enable the finding of another model.
    block = [sol() != model.eval(sol(), model_completion=True) for sol in model]
    solver.add(z3.Or(block))

# convert the solution to a list of states.
stable3= list(map(lambda model: {symbols(str(sol())): bool(model[sol]) for sol in model}, models))

In [ ]:
print(tabulate(stable3, headers='keys', tablefmt='dpsl'))

In [ ]:
print("# Stable states: %d"% len(stable3))

# Test Print

In [3]:
TRACE: bool = True                                                                        #Trace flag for debugging purposes. If True, prints the actions to the console.
def trace(msg: str):
    """
    Prints a trace message to the console if the TRACE flag is set to True.

    :param msg: The message to be printed.
    :type msg: str
    :return: None
    """
    if TRACE:
        print(msg)
        

In [4]:
data = 10000

trace(f"data : {data}")

data : 10000
